# The spectral fractional Laplacian, step by step

The companion notebook made *time* fractional. This one makes *space* fractional,
using the spectral realization

$$
(-\Delta_D)^{s}u=\sum_{k}\lambda_k^{s}\,(u,q_k)_{L^2(\Omega)}\,q_k ,
$$

where $(\lambda_k,q_k)$ are the eigenpairs of the homogeneous-Dirichlet Laplacian
on $\Omega$ and $0<s<1$.

Yonderdrake also provides `RieszFractionalLaplacian`, which applies the
whole-space singular integral to the zero extension of $u$. The two operators
have different eigenfunctions, boundary behaviour, and regularity. Select the
realization specified by the model.

We build up:

1. a graded mesh and the boundary behaviour that motivates it.
2. the operator applied to a known eigenfunction.
3. sinc quadrature and the finite-element discretization floor.
4. solutions of $(-\Delta_D)^{s}u=1$ for several $s$.
5. the effect of mesh grading.
6. coupling back to a fractional time derivative.

Run this in an activated Firedrake environment with Yonderdrake installed.

> **These notebooks run in a single process.** A Jupyter kernel is one process,
> so everything here executes serially however the environment was launched.
>
> For parallel runs, use `mpiexec -n N python your_script.py`, as in the scripts
> under `demos/`.
> Driving MPI from Jupyter via `ipyparallel` is currently untested.

In [ ]:
import firedrake as fd
import matplotlib.pyplot as plt
import numpy as np
from firedrake import (
    Constant,
    DirichletBC,
    Function,
    PointEvaluator,
    SpatialCoordinate,
    TestFunction,
    assemble,
    dx,
    inner,
    norm,
    pi,
    sin,
    solve,
)
from firedrake.pyplot import tripcolor, triplot

from yonderdrake import (
    BirkSong,
    CaputoDerivative,
    FractionalTimeStepper,
    SpectralFractionalLaplacian,
)

plt.rcParams["figure.dpi"] = 110

MATFREE = {
    "snes_type": "ksponly",
    "mat_type": "matfree",
    "ksp_type": "gmres",
    "pc_type": "none",
    "ksp_rtol": 1.0e-8,
}
SHIFT = {"ksp_type": "preonly", "pc_type": "lu"}
print("yonderdrake", __import__("yonderdrake").__version__)

## 1. A graded mesh

Solutions of fractional Dirichlet problems are typically *less* regular at the
boundary than their classical counterparts: they approach zero like a fractional
power of the distance to $\partial\Omega$. A uniform mesh spends its resolution
in the smooth interior and under-resolves exactly the layer that carries the
fractional character.

The same coordinate grading as in the companion notebook fixes that, with no
external mesh generator.

In [ ]:
def graded_unit_square(n, r=2.0):
    "Unit square with cells graded towards the boundary (r=1 is uniform)."
    base = fd.UnitSquareMesh(n, n, diagonal="crossed")
    coordinates = base.coordinates.function_space()
    x, y = SpatialCoordinate(base)

    def grade(xi):
        return fd.conditional(
            fd.le(xi, 0.5),
            0.5 * (2 * xi) ** r,
            1.0 - 0.5 * (2 * (1 - xi)) ** r,
        )

    mapped = Function(coordinates).interpolate(fd.as_vector([grade(x), grade(y)]))
    return fd.Mesh(mapped)


mesh = graded_unit_square(16, r=2.0)
uniform_mesh = graded_unit_square(16, r=1.0)

figure, axes = plt.subplots(1, 2, figsize=(9, 4))
triplot(uniform_mesh, axes=axes[0])
axes[0].set_title("uniform")
triplot(mesh, axes=axes[1])
axes[1].set_title("graded towards the boundary")
for axis in axes:
    axis.set_aspect("equal")
    axis.set_xticks([0, 0.5, 1])
    axis.set_yticks([0, 0.5, 1])
plt.tight_layout()
print(f"cells: {mesh.num_cells()}")

## 2. The operator on a known eigenfunction

On the unit square, $\sin(\pi x)\sin(\pi y)$ is a Dirichlet eigenfunction with
eigenvalue $\lambda = 2\pi^{2}$. By the definition above,

$$(-\Delta_D)^{s}\sin(\pi x)\sin(\pi y)=(2\pi^{2})^{s}\sin(\pi x)\sin(\pi y),$$

which gives an exact target for any $s$. `assemble(operator)` returns the action
as a `Function`. The plot shows its error as $s$ increases.

In [ ]:
V = fd.FunctionSpace(uniform_mesh, "CG", 2)
x, y = SpatialCoordinate(uniform_mesh)
eigenfunction = Function(V).interpolate(sin(pi * x) * sin(pi * y))
boundary = DirichletBC(V, 0.0, "on_boundary")

orders = [0.1, 0.25, 0.4, 0.6, 0.75, 0.9]
action_errors = []
for s in orders:
    operator = SpectralFractionalLaplacian(
        eigenfunction,
        s,
        bcs=boundary,
        sinc_truncation_target=1.0e-10,
        shift_cache="all",
        shift_solver_parameters=SHIFT,
    )
    action = assemble(operator)
    continuum = Function(V).interpolate((2.0 * pi**2) ** s * eigenfunction)
    action_errors.append(norm(action - continuum) / norm(continuum))
    print(f"s = {s:.2f}:  relative action error = {action_errors[-1]:.3e}")

plt.figure(figsize=(5, 3.2))
plt.semilogy(orders, action_errors, "o-")
plt.xlabel("s")
plt.ylabel("relative action error")
plt.title("discrete vs continuum eigenvalue action")
plt.grid(True, alpha=0.3)
plt.tight_layout()

The error climbs steadily with $s$ because larger $s$ weights the high
eigenvalues $\lambda_k^{s}$ more heavily. The finite-element space resolves
those modes least accurately. Mesh refinement reduces this error.

## 3. The sinc quadrature knob, and the floor beneath it

Yonderdrake never forms an eigendecomposition. It evaluates a Balakrishnan
integral representation with **sinc quadrature**, turning each application into a
sum of shifted elliptic solves that stay distributed.

`sinc_truncation_target` controls the sinc-quadrature truncation estimate. Mesh
resolution controls the finite-element error. On a fixed mesh, tighter targets
stop improving the result once that discretization error dominates.

In [ ]:
s = 0.6
targets = [1.0e-2, 1.0e-3, 1.0e-4, 1.0e-6, 1.0e-8, 1.0e-10]
target_errors, node_counts = [], []

for target in targets:
    operator = SpectralFractionalLaplacian(
        eigenfunction,
        s,
        bcs=boundary,
        sinc_truncation_target=target,
        shift_cache="all",
        shift_solver_parameters=SHIFT,
    )
    action = assemble(operator)
    continuum = Function(V).interpolate((2.0 * pi**2) ** s * eigenfunction)
    target_errors.append(norm(action - continuum) / norm(continuum))
    node_counts.append(operator.diagnostics()["num_nodes"])
    print(
        f"target {target:.0e}: {node_counts[-1]:3d} sinc nodes -> "
        f"error {target_errors[-1]:.3e}"
    )

figure, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].loglog(targets, target_errors, "o-")
axes[0].set_xlabel("sinc_truncation_target")
axes[0].set_ylabel("relative action error")
axes[0].set_title("accuracy plateaus at the mesh")
axes[0].grid(True, which="both", alpha=0.3)
axes[1].semilogx(targets, node_counts, "o-")
axes[1].set_xlabel("sinc_truncation_target")
axes[1].set_ylabel("sinc nodes (cost)")
axes[1].set_title("cost keeps rising")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()

The left curve reaches a plateau. The right curve continues to decrease under
mesh refinement. Beyond the left-hand plateau, additional shifted solves do not
improve the result.

## 4. Solving a fractional Dirichlet problem

Now use the operator inside a residual and solve

$$(-\Delta_D)^{s}u=1 \quad\text{in }\Omega,\qquad u=0\text{ on }\partial\Omega.$$

The operator supplies a matrix-free Jacobian action, so we drive it with a
matrix-free GMRES.

In [ ]:
graded_space = fd.FunctionSpace(mesh, "CG", 1)
test = TestFunction(graded_space)
graded_bc = DirichletBC(graded_space, 0.0, "on_boundary")

solutions = {}
for s in (0.3, 0.6, 0.9):
    field = Function(graded_space, name=f"u_s{s}")
    operator = SpectralFractionalLaplacian(
        field,
        s,
        bcs=graded_bc,
        sinc_truncation_target=1.0e-6,
        shift_cache="all",
        shift_solver_parameters=SHIFT,
    )
    residual = (inner(operator, test) - inner(Constant(1.0), test)) * dx
    solve(residual == 0, field, bcs=graded_bc, solver_parameters=MATFREE)
    solutions[s] = field
    print(f"s = {s}: peak value {field.dat.data_ro.max():.4f}")

figure, axes = plt.subplots(1, len(solutions), figsize=(3.3 * len(solutions), 3.1))
for axis, (s, field) in zip(axes, solutions.items(), strict=False):
    contours = tripcolor(field, axes=axis, cmap="viridis")
    axis.set_title(f"s = {s}")
    axis.set_aspect("equal")
    axis.set_xticks([])
    axis.set_yticks([])
    figure.colorbar(contours, ax=axis, fraction=0.046)
plt.tight_layout()

Smaller $s$ means a weaker operator, so the same unit load produces a larger
response. Take a slice to see the boundary behaviour directly.

In [ ]:
samples = np.linspace(0.001, 0.999, 400)
slice_points = np.column_stack([samples, np.full_like(samples, 0.5)])
evaluator = PointEvaluator(mesh, slice_points)

plt.figure(figsize=(6, 3.4))
for s, field in solutions.items():
    values = np.asarray(evaluator.evaluate(field))
    plt.plot(samples, values / values.max(), label=f"s = {s}")
plt.xlabel("x  (slice at y = 0.5)")
plt.ylabel("u / max u")
plt.title("normalized profiles: smaller s is blunter at the wall")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

## 5. Does the grading actually help?

A fair test holds the cost fixed. We compare a uniform and a graded mesh with
essentially the same number of degrees of freedom, and measure against a
well-resolved reference computed on a much finer graded mesh.

In [ ]:
s = 0.6


def solve_unit_load(target_mesh, degree=1):
    space = fd.FunctionSpace(target_mesh, "CG", degree)
    field = Function(space)
    test_function = TestFunction(space)
    bc = DirichletBC(space, 0.0, "on_boundary")
    operator = SpectralFractionalLaplacian(
        field,
        s,
        bcs=bc,
        sinc_truncation_target=1.0e-6,
        shift_cache="all",
        shift_solver_parameters=SHIFT,
    )
    form = (inner(operator, test_function) - inner(Constant(1.0), test_function)) * dx
    solve(form == 0, field, bcs=bc, solver_parameters=MATFREE)
    return field


reference = solve_unit_load(graded_unit_square(48, r=2.0))
coarse_uniform = solve_unit_load(graded_unit_square(16, r=1.0))
coarse_graded = solve_unit_load(graded_unit_square(16, r=2.0))

probe = np.linspace(0.002, 0.2, 60)
probe_points = np.column_stack([probe, np.full_like(probe, 0.5)])


def profile(field):
    "Sample a field along y = 0.5 near the left wall."
    mesh_of = field.function_space().mesh()
    return np.asarray(PointEvaluator(mesh_of, probe_points).evaluate(field))


reference_profile = profile(reference)
errors = {
    "uniform": np.abs(profile(coarse_uniform) - reference_profile),
    "graded": np.abs(profile(coarse_graded) - reference_profile),
}

print(
    f"degrees of freedom: uniform {coarse_uniform.function_space().dim()}, "
    f"graded {coarse_graded.function_space().dim()}"
)
for label, value in errors.items():
    print(f"{label:8s}: max near-boundary error {value.max():.3e}")

plt.figure(figsize=(6, 3.4))
for label, value in errors.items():
    plt.semilogy(probe, value, label=label)
plt.xlabel("distance from the wall along y = 0.5")
plt.ylabel("|error| vs fine reference")
plt.title(f"near-boundary accuracy at equal cost, s = {s}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()

## 6. Fractional in time *and* space

The spectral operator is an ordinary term in the residual, and the Caputo marker
wraps the
stepped field as before.

$$D_C^{\alpha}u + (-\Delta_D)^{s}u = 0,\qquad u(\cdot,0)=\sin(\pi x)\sin(\pi y).$$

Because the initial condition is an eigenfunction, the exact solution stays
proportional to it and decays like $E_\alpha\!\left(-(2\pi^{2})^{s}t^{\alpha}\right)$.
This gives a closed-form check for the coupled solve.

In [ ]:
import mpmath as mp

mp.mp.dps = 30


def mittag_leffler(order, z, terms=400):
    return float(sum(mp.mpf(z) ** k / mp.gamma(order * k + 1) for k in range(terms)))


alpha, s = 0.7, 0.5
space = fd.FunctionSpace(uniform_mesh, "CG", 1)
xx, yy = SpatialCoordinate(uniform_mesh)
shape = sin(pi * xx) * sin(pi * yy)
field = Function(space).interpolate(shape)
test_function = TestFunction(space)
clock, increment = Constant(0.0), Constant(0.005)
bc = DirichletBC(space, 0.0, "on_boundary")

spatial = SpectralFractionalLaplacian(
    field,
    s,
    bcs=bc,
    sinc_truncation_target=1.0e-6,
    shift_cache="all",
    shift_solver_parameters=SHIFT,
)
coupled = (
    inner(CaputoDerivative(field, alpha), test_function) + inner(spatial, test_function)
) * dx
stepper = FractionalTimeStepper(
    coupled,
    BirkSong(32),
    clock,
    increment,
    field,
    bcs=bc,
    solver_parameters=MATFREE,
)

peak = Function(space).interpolate(shape).dat.data_ro.max()
history = [(0.0, 1.0)]
for _ in range(40):
    stepper.advance()
    clock.assign(clock + increment)
    history.append((float(clock), field.dat.data_ro.max() / peak))

history = np.array(history)
eigenvalue = (2.0 * pi**2) ** s
exact = np.array(
    [
        mittag_leffler(alpha, -eigenvalue * tt**alpha) if tt > 0 else 1.0
        for tt in history[:, 0]
    ]
)

plt.figure(figsize=(6, 3.4))
plt.plot(history[:, 0], exact, "k-", label="exact Mittag-Leffler decay")
plt.plot(history[:, 0], history[:, 1], "o", ms=4, label="computed")
plt.xlabel("t")
plt.ylabel("amplitude / initial amplitude")
plt.title(f"coupled decay, alpha = {alpha}, s = {s}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

print(f"max relative amplitude error: {np.max(np.abs(history[:, 1] - exact)):.3e}")
report = spatial.diagnostics()
for key in (
    "num_nodes",
    "effective_truncation_target",
    "estimated_model_error",
    "shift_solves",
    "cache_reuses",
):
    print(f"{key:30s} {report[key]}")

The amplitude follows the Mittag-Leffler curve. The residual gap matches the
spatial eigenvalue-discretization error measured in step 2.

## Where to go next

- `RieszFractionalLaplacian` is the other realization. Solve the same unit-load
  problem with it and compare the boundary profiles of the two realizations.
- `PeriodicFractionalLaplacian` is a third realization on periodic geometry, so its use is more restricted.
- `shift_cache="stream"` bounds memory for meshes whose shifted solutions do
  not fit comfortably in memory.
- The library documentation covers method selection, the defaults policy, and how
  each operator relates to its source paper.